# Forex Trading Strategy Analysis

This notebook analyzes EUR/USD Forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [137]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import sys
import importlib

# Force reload of utils module to prevent caching
if 'utils' in sys.modules:
    del sys.modules['utils']

# Enable automatic reloading of modules
%load_ext autoreload
%autoreload 2

# Import utility functions (will be freshly loaded)
import utils
from utils import (
    load_and_clean_data, 
    analyze_entry_timing, 
    display_profitable_strategies,
    Strategy,
    create_strategy_library,
    evaluate_all_strategies,
    get_top_strategies_by_edge,
    analyze_pullback_profitability
)

# Force reload utils to ensure latest version
importlib.reload(utils)

# Load the data
df = load_and_clean_data()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [138]:
# Display entry timing analysis with detailed tables
from utils import analyze_entry_timing_detailed

display(HTML("""
    <h2>What type of entry is the best?</h2>
"""))

# Get the detailed entry timing tables
entry_tables = analyze_entry_timing_detailed(df)

# Display each table with styling similar to Profitable Strategies
for method_name, table_df in entry_tables.items():
    # Style the DataFrame for better readability
    first_column = table_df.columns[0]
    styled_df = table_df.style.set_properties(
        subset=[first_column], 
        **{'width': '250px', 'font-weight': 'bold'}
    ) 
    display(styled_df)
    print()  # Add spacing between tables

display(HTML("""
    <p><b>Insights</b></p>
    <ul>
        <li>No clear winner.</li>
        <li>1M Confirmation Candle gives the most trades (bad) and also almost -10% disadvantage.</li>
    </ul>
"""))

,5M Stop,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,89,89,89
1,Wins,40,26,16
2,Losses,49,63,73
3,Win Rate,44.9%,29.2%,18.0%
4,Edge,-5.1%,-4.1%,-7.0%
5,Outcome,-9R,-11R,-25R


,5M Breakout,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,92,92,92
1,Wins,40,26,18
2,Losses,52,66,74
3,Win Rate,43.5%,28.3%,19.6%
4,Edge,-6.5%,-5.0%,-5.4%
5,Outcome,-12R,-14R,-20R


,5M Confirmation Candle,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,97,97,97
1,Wins,41,27,20
2,Losses,56,70,77
3,Win Rate,42.3%,27.8%,20.6%
4,Edge,-7.7%,-5.5%,-4.4%
5,Outcome,-15R,-16R,-17R


,1M Confirmation Candle,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,140,140,140
1,Wins,50,33,23
2,Losses,90,107,117
3,Win Rate,35.7%,23.6%,16.4%
4,Edge,-14.3%,-9.7%,-8.6%
5,Outcome,-40R,-41R,-48R


In [139]:
# Pullback Analysis
display(HTML("<h2>How does pullback size affect profitability?</h2>"))

# Get the pullback analysis tables
pullback_tables = analyze_pullback_profitability(df)

# Display each table with styling similar to "What type of entry is the best?"
for pullback_name, table_df in pullback_tables.items():
    # Style the DataFrame for better readability
    first_column = table_df.columns[0]
    styled_df = table_df.style.set_properties(
        subset=[first_column], 
        **{'width': '250px', 'font-weight': 'bold'}
    ) 
    display(styled_df)
    print()  # Add spacing between tables

display(HTML("""
    <p><b>Insights</b></p>
    <ul>
        <li>As pullback size increases, win rate and edge decrease.</li>
        <li>Pullback should not be traded.</li>
    </ul>
"""))

,No Pullback,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,140,140,140
1,Wins,50,33,23
2,Losses,90,107,117
3,Win Rate,35.7%,23.6%,16.4%
4,Edge,-14.3%,-9.7%,-8.6%
5,Outcome,-40R,-41R,-48R


,Pullback >= 0.5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,133,133,133
1,Wins,45,31,21
2,Losses,88,102,112
3,Win Rate,33.8%,23.3%,15.8%
4,Edge,-16.2%,-10.0%,-9.2%
5,Outcome,-43R,-40R,-49R


,Pullback >= 1.0 pip,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,124,124,124
1,Wins,36,23,15
2,Losses,88,101,109
3,Win Rate,29.0%,18.5%,12.1%
4,Edge,-21.0%,-14.8%,-12.9%
5,Outcome,-52R,-55R,-64R


,Pullback >= 1.5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,115,115,115
1,Wins,30,19,12
2,Losses,85,96,103
3,Win Rate,26.1%,16.5%,10.4%
4,Edge,-23.9%,-16.8%,-14.6%
5,Outcome,-55R,-58R,-67R


,Pullback >= 2.0 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,97,97,97
1,Wins,21,15,9
2,Losses,76,82,88
3,Win Rate,21.6%,15.5%,9.3%
4,Edge,-28.4%,-17.8%,-15.7%
5,Outcome,-55R,-52R,-61R


In [140]:
# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

# Add all strategies from the library
strategies.extend(create_strategy_library())

# Evaluate all strategies
strategy_results = evaluate_all_strategies(df, strategies)

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

# Display top performing strategies for each RRR - sorted by Edge
display(HTML("<h2>🎯 TOP Performing Strategies</h2>"))

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies by Edge</h3>"))
    
    # Get and display top strategies sorted by edge
    top_df = get_top_strategies_by_edge(strategy_results, rrr_column)
    top_df = top_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px'}
    ).set_properties(
        subset=['Edge'],
        **{'color': 'green'}
    )
    
    display(styled_df)
    print()  # Add spacing

display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>EMA + BOS is what I have traded for a lot of months now.</li>
    </ul>
"""))

,Top 1:1 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,30M Trend + 5 < SL < 15,1M CC,28,16,12,57.1%,7.1%,4R
1,30M Trend + 5 < SL < 15,5M CC,28,16,12,57.1%,7.1%,4R
2,30M Trend + 5 < SL < 15,5M Breakout,28,16,12,57.1%,7.1%,4R
3,30M Trend + SL > 5 pips,1M CC,29,16,13,55.2%,5.2%,3R
4,30M Trend + SL > 5 pips,5M CC,29,16,13,55.2%,5.2%,3R
5,30M Trend + SL > 5 pips,5M Breakout,29,16,13,55.2%,5.2%,3R
6,Aggressive: SL >= 7 pips,1M CC,24,13,11,54.2%,4.2%,2R
7,30M Trend + 5 < SL < 15,5M Stop,28,15,13,53.6%,3.6%,2R
8,30M Trend + BOS + SL < 10,1M CC,58,31,27,53.4%,3.4%,4R
9,30M Trend + News > 2hrs,1M CC,19,10,9,52.6%,2.6%,1R


,Top 1:2 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,10,5,5,50.0%,16.7%,5R
1,30M Trend + BOS + SL < 10,1M CC,58,24,34,41.4%,8.1%,14R
2,30M Trend + BOS + SL < 10,5M Stop,58,24,34,41.4%,8.1%,14R
3,30M Trend + BOS,1M CC,60,24,36,40.0%,6.7%,12R
4,BOS + Conservative SL,5M CC,10,4,6,40.0%,6.7%,2R
5,BOS + Conservative SL,5M Stop,10,4,6,40.0%,6.7%,2R
6,30M Trend + BOS,5M Stop,60,24,36,40.0%,6.7%,12R
7,BOS + Conservative SL,5M Breakout,10,4,6,40.0%,6.7%,2R
8,30M Trend + BOS + SL < 10,5M CC,58,23,35,39.7%,6.4%,11R
9,30M Trend + 5 < SL < 15,5M Breakout,28,11,17,39.3%,6.0%,5R


,Top 1:3 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,10,5,5,50.0%,25.0%,10R
1,BOS + Conservative SL,5M Stop,10,4,6,40.0%,15.0%,6R
2,BOS + Conservative SL,5M Breakout,10,4,6,40.0%,15.0%,6R
3,Conservative: SL <= 2 pips,1M CC,14,5,9,35.7%,10.7%,6R
4,30M Trend + BOS + SL < 10,1M CC,58,19,39,32.8%,7.8%,18R
5,30M Trend + EMA + BOS + SL < 10,1M CC,50,16,34,32.0%,7.0%,14R
6,30M Trend + BOS,1M CC,60,19,41,31.7%,6.7%,16R
7,30M Trend + News > 2hrs,1M CC,19,6,13,31.6%,6.6%,5R
8,30M Trend + News > 2hrs,5M CC,19,6,13,31.6%,6.6%,5R
9,30M Trend + News > 2hrs,5M Breakout,19,6,13,31.6%,6.6%,5R


In [141]:
# Display only profitable strategies
display_profitable_strategies(strategy_results)

,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,24,24,24
1,Wins,13,6,4
2,Losses,11,18,20
3,Win Rate,54.2%,25.0%,16.7%
4,Edge,4.2%,-8.3%,-8.3%
5,Outcome,2R,-6R,-8R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + BOS,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,60,60,60
1,Wins,31,24,19
2,Losses,29,36,41
3,Win Rate,51.7%,40.0%,31.7%
4,Edge,1.7%,6.7%,6.7%
5,Outcome,2R,12R,16R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + SL > 5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,29,29,29
1,Wins,16,10,6
2,Losses,13,19,23
3,Win Rate,55.2%,34.5%,20.7%
4,Edge,5.2%,1.2%,-4.3%
5,Outcome,3R,1R,-5R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + 3 < SL < 10,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,59,59,59
1,Wins,30,19,12
2,Losses,29,40,47
3,Win Rate,50.8%,32.2%,20.3%
4,Edge,0.8%,-1.1%,-4.7%
5,Outcome,1R,-2R,-11R
6,Entry,1M CC,1M CC,1M CC


,30M Trend + 5 < SL < 15,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,28,28,28
1,Wins,16,10,6
2,Losses,12,18,22
3,Win Rate,57.1%,35.7%,21.4%
4,Edge,7.1%,2.4%,-3.6%
5,Outcome,4R,2R,-4R
6,Entry,1M CC,1M CC,1M CC


...

,30M Trend + 5 < SL < 15,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,28,28,28
1,Wins,16,10,7
2,Losses,12,18,21
3,Win Rate,57.1%,35.7%,25.0%
4,Edge,7.1%,2.4%,0.0%
5,Outcome,4R,2R,0R
6,Entry,5M CC,5M CC,5M CC


,30M Trend + SL > 5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,29,29,29
1,Wins,15,10,5
2,Losses,14,19,24
3,Win Rate,51.7%,34.5%,17.2%
4,Edge,1.7%,1.2%,-7.8%
5,Outcome,1R,1R,-9R
6,Entry,5M Stop,5M Stop,5M Stop


,30M Trend + 5 < SL < 15,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,28,28,28
1,Wins,15,10,5
2,Losses,13,18,23
3,Win Rate,53.6%,35.7%,17.9%
4,Edge,3.6%,2.4%,-7.1%
5,Outcome,2R,2R,-8R
6,Entry,5M Stop,5M Stop,5M Stop


,30M Trend + SL > 5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,29,29,29
1,Wins,16,11,6
2,Losses,13,18,23
3,Win Rate,55.2%,37.9%,20.7%
4,Edge,5.2%,4.6%,-4.3%
5,Outcome,3R,4R,-5R
6,Entry,5M Breakout,5M Breakout,5M Breakout


,30M Trend + 5 < SL < 15,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,28,28,28
1,Wins,16,11,6
2,Losses,12,17,22
3,Win Rate,57.1%,39.3%,21.4%
4,Edge,7.1%,6.0%,-3.6%
5,Outcome,4R,5R,-4R
6,Entry,5M Breakout,5M Breakout,5M Breakout
